# Pelabelan data

In [10]:
import pandas as pd

df = pd.read_csv("data_ulasan_mentah.csv")

def label(score):
  if score <= 2:
    return "negatif"
  elif score == 3:
    return "netral"
  else:
    return "positif"

df["label"] = df["score"].apply(label)

print("Jumlah data per kelas:")
print(df["label"].value_counts())

Jumlah data per kelas:
label
positif    7082
negatif    3172
netral      746
Name: count, dtype: int64


# Preprocessing Data

In [11]:
import re
import string

df['content'] = df['content'].fillna('').astype(str)

def clean_text(text):

  if not isinstance(text, str):
    return ""

  text = text.lower()
  text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
  text = re.sub(r"\@\w+|\#", "", text)
  text = re.sub(r"[^a-zA-Z\s]", "", text)
  text = re.sub(r"\s+", " ", text).strip()
  return text

df["text_clean"] = df["content"].apply(clean_text)
df = df[df["text_clean"] != ""]
df[['content', 'text_clean']].head()

,content,text_clean
0,game nya seru banget permainan nya juga banyak...,game nya seru banget permainan nya juga banyak...
1,tolong kasih fitur chat,tolong kasih fitur chat
2,Sering terjadi server error jadi tidak terlalu...,sering terjadi server error jadi tidak terlalu...
3,bagus banget aku ngak bisa berkata² pokok debe...,bagus banget aku ngak bisa berkata pokok debes...
4,KALO MAU BINTANG LIMA BENERIN DULU NOH GAME LUU🙄,kalo mau bintang lima benerin dulu noh game luu


# Normalisasi

In [15]:
slang_dict = {
    "gx": "tidak",
    "gk": "tidak",
    "gak": "tidak",
    "ngak": "tidak",
    "nggak": "tidak",
    "bgt": "banget",
    "bgs": "bagus",
    "bgus": "bagus",
    "game ny": "gamenya",
    "game nya": "gamenya",
    "gamnya": "gamenya",
    "gem": "game",
    "klo": "kalau",
    "udh": "sudah",
    "sdh": "sudah",
    "pake": "pakai",
    "aj": "saja",
    "aja": "saja",
    "gw": "saya",
    "gua": "saya",
    "donk": "dong",
    "yaudah": "ya sudah",
}

def normalisasi_kata(text):
    words = text.split()
    normalized_words = [slang_dict.get(w, w) for w in words]
    return " ".join(normalized_words)

# Terapkan ke data yang sudah bersih tadi
df['text_normalized'] = df['text_clean'].apply(normalisasi_kata)

print("Hasil Normalisasi:")
df[['text_clean', 'text_normalized']].head()

Hasil Normalisasi:


,text_clean,text_normalized
0,game nya seru banget permainan nya juga banyak...,game nya seru banget permainan nya juga banyak...
1,tolong kasih fitur chat,tolong kasih fitur chat
2,sering terjadi server error jadi tidak terlalu...,sering terjadi server error jadi tidak terlalu...
3,bagus banget aku ngak bisa berkata pokok debes...,bagus banget aku tidak bisa berkata pokok debe...
4,kalo mau bintang lima benerin dulu noh game luu,kalo mau bintang lima benerin dulu noh game luu


# Label Encoding

In [16]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Positif -> 2, Netral -> 1, Negatif -> 0 (tergantung urutan abjad)
df['label_encoded'] = le.fit_transform(df['label'])

print("Mapping Label:")
for index, class_label in enumerate(le.classes_):
    print(f"{class_label} -> {index}")

Mapping Label:
negatif -> 0
netral -> 1
positif -> 2


# Data Splitting

In [17]:
from sklearn.model_selection import train_test_split


X = df['text_normalized']
y = df['label_encoded']

#80% training 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Total Data Training: {len(X_train)}")
print(f"Total Data Testing: {len(X_test)}")

Total Data Training: 8714
Total Data Testing: 2179


# Oversampling